# M2M GPU Server — Electric Piano, Clean Guitar, Bass & Drums

Electric Piano, Electric Guitar (clean), Electric Bass, and Drums

# GPU pod setup (run these commands in a local PowerShell terminal, not in a notebook cell)

This is a headless CUDA pod. The current image has CUDA but does **not** include Python, pip, or Git.
Run the one-time bootstrap below, then copy this notebook and the MIDI files into the pod.
The notebook installs PyTorch and its CUDA 12.1 wheels with pip caching disabled to stay within the namespace's 10 GiB ephemeral-storage limit.

```powershell
$ns = "<your-namespace>"
$pod = "<your-gpu-pod>"
kubectl exec -n $ns $pod -- bash -lc 'apt-get update -qq && apt-get install --no-install-recommends -y python3 python3-venv python3-pip git && apt-get clean && rm -rf /var/lib/apt/lists/* && mkdir -p /workspace/m2m/inputs /workspace/m2m/outputs'
kubectl exec -n $ns $pod -- bash -lc 'python3 -m venv /workspace/m2m/venv && /workspace/m2m/venv/bin/pip install --no-cache-dir jupyter nbconvert ipykernel'
Get-ChildItem .\corrected_midi\*.mid | ForEach-Object {
    $relativePath = Join-Path 'corrected_midi' $_.Name
    kubectl cp $relativePath "${ns}/${pod}:/workspace/m2m/inputs/$($_.Name)"
}
kubectl cp .\ml\m2m\notebooks\NOTEBOOK_FILE.ipynb ${ns}/${pod}:/workspace/m2m/NOTEBOOK_FILE.ipynb
kubectl exec -n $ns $pod -- bash -lc 'PATH=/workspace/m2m/venv/bin:$PATH jupyter nbconvert --to notebook --execute /workspace/m2m/NOTEBOOK_FILE.ipynb --output NOTEBOOK_FILE.executed.ipynb --output-dir /workspace/m2m --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3'
kubectl cp ${ns}/${pod}:/workspace/m2m/outputs .\m2m_outputs
```

Run this from the repository root, with input MIDI files in the local `corrected_midi` directory. Replace `NOTEBOOK_FILE` with the selected name. Every `.mid`/`.midi` file in `/workspace/m2m/inputs` is processed in one run. Outputs and Hugging Face model caches are under `/workspace/m2m`; copy anything important back before the pod is deleted.


In [ ]:
# Install the GPU runtime dependencies and check out the model code.
import os
import shutil
import subprocess
import sys
from pathlib import Path

SERVER_ROOT = Path("/workspace/m2m")
M2M_REPO = SERVER_ROOT / "music2music_code"
REMI_REPO = SERVER_ROOT / "REMI-z"
SERVER_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(SERVER_ROOT / "cache" / "huggingface")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

# --no-cache-dir prevents pip from retaining the large CUDA wheel downloads.
%pip -q install --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://pypi.org/simple "numpy<2" "torch==2.2.2+cu121" "transformers==4.40.0" mido tqdm "git+https://github.com/Sonata165/REMI-z.git"

if not M2M_REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Sonata165/music2music_code.git", str(M2M_REPO)],
        check=True,
    )
if not REMI_REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Sonata165/REMI-z.git", str(REMI_REPO)],
        check=True,
    )

# Some REMI-z releases look up YAML resources beside the installed package.
import remi_z as _installed_remiz

installed_remiz_dir = Path(_installed_remiz.__file__).resolve().parent
for yaml_file in (REMI_REPO / "remi_z").glob("*.yaml"):
    shutil.copy2(yaml_file, installed_remiz_dir / yaml_file.name)

if str(M2M_REPO) not in sys.path:
    sys.path.insert(0, str(M2M_REPO))

import torch

assert torch.cuda.is_available(), "This notebook must run in the GPU pod with CUDA available."
print("GPU:", torch.cuda.get_device_name(0))
print("M2M commit:", subprocess.check_output(["git", "-C", str(M2M_REPO), "rev-parse", "HEAD"], text=True).strip())
print("REMI-z commit:", subprocess.check_output(["git", "-C", str(REMI_REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
# Arrangement configuration for this notebook.
from api.arranger import BandArranger, DrumArranger
from remi_z import Bar, MultiTrack, Track

ARRANGEMENT_NAME = 'Electric Piano, Electric Guitar (clean), Electric Bass, and Drums'
CONDITION_PROGRAMS = [2, 26, 33]
OUTPUT_PROGRAMS = [4, 27, 33, 128]
PROGRAM_REMAP = {2: 4, 26: 27}
PROGRAM_LABELS = {4: 'Electric Piano 1', 27: 'Electric Guitar (clean)', 33: 'Electric Bass (finger)', 128: 'Drums'}
OUTPUT_SUFFIX = 'epiano_clean_guitar_bass_drum'

BASE_SEED = 20261017
MAX_ATTEMPTS = 3
BAND_MODEL_ID = "LongshenOu/m2m_arranger"
DRUM_MODEL_ID = "LongshenOu/m2m_drummer"

print("Arrangement:", ARRANGEMENT_NAME)
print("Model conditioning IDs:", CONDITION_PROGRAMS)
print("Saved MIDI IDs:", OUTPUT_PROGRAMS)
for program in OUTPUT_PROGRAMS:
    print(f"  {program:>3}  {PROGRAM_LABELS[program]}")


In [ ]:
# Process all MIDI files staged in the GPU pod.
from pathlib import Path

INPUT_DIR = Path("/workspace/m2m/inputs")
INPUT_PATHS = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".mid", ".midi"}
)
assert INPUT_PATHS, f"No MIDI files found in {INPUT_DIR}"

for input_path in INPUT_PATHS:
    print(f"Queued: {input_path.name} ({input_path.stat().st_size:,} bytes)")


In [ ]:
# Helpers: reproducibility, validation, exact GM relabeling, and safe drum merging.
import gc
import hashlib
import json
import random
import zipfile
from datetime import datetime, timezone

import numpy as np


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def note_counts(multitrack):
    counts = {}
    for bar in multitrack.bars:
        for program, track in bar.tracks.items():
            counts[program] = counts.get(program, 0) + len(track.notes)
    return counts


def rebuild_bar(bar, grouped_notes):
    tracks = [
        Track.from_note_list(program, list(grouped_notes[program]))
        for program in sorted(grouped_notes)
        if grouped_notes[program]
    ]
    return Bar.from_tracks(
        bar.bar_id,
        tracks,
        time_signature=bar.time_signature,
        tempo=bar.tempo,
    )


def remap_programs(multitrack, mapping):
    bars = []
    for bar in multitrack.bars:
        grouped = {}
        for program, track in bar.tracks.items():
            target = mapping.get(program, program)
            grouped.setdefault(target, []).extend(track.notes)
        bars.append(rebuild_bar(bar, grouped))
    return MultiTrack.from_bars(bars)


def merge_band_and_drums(band, drums):
    """Keep the band's exact length and take program 128 only from the drummer."""
    bars = []
    for index, band_bar in enumerate(band.bars):
        grouped = {
            program: list(track.notes)
            for program, track in band_bar.tracks.items()
            if program != 128
        }
        if index < len(drums.bars) and 128 in drums.bars[index].tracks:
            grouped.setdefault(128, []).extend(drums.bars[index].tracks[128].notes)
        bars.append(rebuild_bar(band_bar, grouped))
    return MultiTrack.from_bars(bars)


def require_programs(multitrack, programs, label):
    counts = note_counts(multitrack)
    missing = [program for program in programs if counts.get(program, 0) == 0]
    if missing:
        raise RuntimeError(f"{label} has empty/missing programs: {missing}; counts={counts}")
    return counts


def keep_only_programs(multitrack, allowed_programs):
    allowed = set(allowed_programs)
    return MultiTrack.from_bars([
        rebuild_bar(bar, {
            program: list(track.notes)
            for program, track in bar.tracks.items()
            if program in allowed
        })
        for bar in multitrack.bars
    ])


In [ ]:
# Generate all pitched parts first, then drums, and save validated combined MIDIs.
WORK_DIR = Path("/workspace/m2m/outputs/epiano_clean_guitar_bass_drum")
WORK_DIR.mkdir(parents=True, exist_ok=True)

records = []
band_jobs = []
band_model = BandArranger(BAND_MODEL_ID, hf_ckpt=True, device="cuda")

for input_index, input_path in enumerate(INPUT_PATHS):
    print(f"\n=== Band arrangement: {input_path.name} ===")
    last_error = None
    for attempt in range(MAX_ATTEMPTS):
        seed = BASE_SEED + input_index * 100 + attempt
        try:
            seed_everything(seed)
            band = band_model.arrange(
                str(input_path),
                use_preset=None,
                instrument_and_voice=CONDITION_PROGRAMS,
            )
            band_counts = require_programs(band, CONDITION_PROGRAMS, "Band arrangement")
            canonical_path = WORK_DIR / f"{input_path.stem}.band.canonical.mid"
            band.to_midi(str(canonical_path))
            band_jobs.append(
                {
                    "input_path": input_path,
                    "band": band,
                    "canonical_path": canonical_path,
                    "seed": seed,
                    "attempt": attempt + 1,
                    "band_counts": band_counts,
                }
            )
            print("Band note counts:", band_counts)
            break
        except Exception as exc:
            last_error = repr(exc)
            print(f"Attempt {attempt + 1}/{MAX_ATTEMPTS} failed: {last_error}")
    else:
        records.append({"input": input_path.name, "status": "failed_band", "error": last_error})

del band_model
gc.collect()
torch.cuda.empty_cache()

drum_model = DrumArranger(DRUM_MODEL_ID, hf_ckpt=True, device="cuda")

for job_index, job in enumerate(band_jobs):
    input_path = job["input_path"]
    print(f"\n=== Drum arrangement: {input_path.name} ===")
    last_error = None
    for attempt in range(MAX_ATTEMPTS):
        drum_seed = job["seed"] + 10_000 + attempt
        try:
            seed_everything(drum_seed)
            drums = drum_model.arrange(
                str(job["canonical_path"]),
                use_preset="drum",
                merge_with_input=False,
            )
            require_programs(drums, [128], "Drum arrangement")
            combined = merge_band_and_drums(job["band"], drums)
            combined = keep_only_programs(remap_programs(combined, PROGRAM_REMAP), OUTPUT_PROGRAMS)
            final_counts = require_programs(combined, OUTPUT_PROGRAMS, "Final arrangement")

            output_path = WORK_DIR / f"{input_path.stem}.{OUTPUT_SUFFIX}.mid"
            combined.to_midi(str(output_path))
            verified_counts = require_programs(
                MultiTrack.from_midi(str(output_path)), OUTPUT_PROGRAMS, "Saved MIDI"
            )
            record = {
                "input": input_path.name,
                "input_sha256": sha256_file(input_path),
                "output": output_path.name,
                "output_sha256": sha256_file(output_path),
                "status": "ok",
                "bars": len(combined.bars),
                "band_seed": job["seed"],
                "band_attempt": job["attempt"],
                "drum_seed": drum_seed,
                "drum_attempt": attempt + 1,
                "conditioning_programs": CONDITION_PROGRAMS,
                "output_programs": OUTPUT_PROGRAMS,
                "program_remap": PROGRAM_REMAP,
                "note_counts": verified_counts,
            }
            records.append(record)
            print("Saved:", output_path)
            print("Final note counts:", final_counts)
            break
        except Exception as exc:
            last_error = repr(exc)
            print(f"Attempt {attempt + 1}/{MAX_ATTEMPTS} failed: {last_error}")
    else:
        records.append({"input": input_path.name, "status": "failed_drum", "error": last_error})

del drum_model
gc.collect()
torch.cuda.empty_cache()

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "arrangement": ARRANGEMENT_NAME,
    "band_model": BAND_MODEL_ID,
    "drum_model": DRUM_MODEL_ID,
    "base_seed": BASE_SEED,
    "records": records,
}
manifest_path = WORK_DIR / f"{OUTPUT_SUFFIX}.manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")

successful_outputs = [WORK_DIR / item["output"] for item in records if item.get("status") == "ok"]
assert successful_outputs, "No output MIDI passed validation. Inspect the errors above."

zip_path = WORK_DIR / f"{OUTPUT_SUFFIX}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(manifest_path, manifest_path.name)
    for output_path in successful_outputs:
        archive.write(output_path, output_path.name)

print(f"\nCompleted {len(successful_outputs)}/{len(INPUT_PATHS)} input files.")
print("Archive:", zip_path)

assert len(successful_outputs) == len(INPUT_PATHS), records


In [ ]:
# Results remain on the pod; copy the printed archive path back before deleting the pod.
print(f"Completed {len(successful_outputs)}/{len(INPUT_PATHS)} input file(s).")
print("Output directory:", WORK_DIR)
print("Archive:", zip_path)
print("Copy it to the local machine with:")
print(f"kubectl cp <your-namespace>/<your-gpu-pod>:{zip_path} .\\{zip_path.name}")
